# ShadowKV++ Semantic Fidelity Measurement

Measures output quality of approximate semantic KV reuse by comparing generated text from exact-prefix vs semantic-approximate KV caches.

**Hardware:** T4 GPU (free Colab tier)
**Models:** gpt2, tinyllama, Qwen2.5-1.5B, Gemma-2B, Phi-3-mini
**Datasets:** All 10 from the paper (ag_news, banking77, alpaca_eval, dolly, daily_dialog, oasst1, ultrachat, samsum, xsum, cnn_dailymail)
**Samples per cell:** 32
**Max gen tokens:** 64

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets==2.21.0 accelerate huggingface-hub rouge-score bert-score numpy

# Login to HuggingFace (needed for gated models like Gemma-2B)
from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')

# Clone the repo
!git clone https://github.com/Kushalk0677/shadowkv.git
%cd shadowkv/experiments
import os
os.makedirs('fidelity_results', exist_ok=True)

# Check T4 GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Run Experiment

Each model runs on all 10 datasets with 32 samples each. Results are saved and downloaded per model.

In [ ]:
import sys, json, time
sys.path.insert(0, '..')

from run_semantic_fidelity import P100_MODELS, DATASETS, run_fidelity_experiment

class Args:
    device = 'cuda:0'
    models = ['gpt2', 'tinyllama', 'qwen25_15b', 'gemma2b', 'phi3mini']
    datasets = ['samsum', 'xsum', 'cnn_dailymail', 'ag_news', 'banking77',
                'alpaca_eval', 'dolly', 'daily_dialog', 'oasst1', 'ultrachat']
    n_samples = 32
    max_gen_tokens = 64
    approximation = 'prefix_shuffle'
    output_dir = 'fidelity_results'

args = Args()
total = len(args.models) * len(args.datasets) * args.n_samples
print(f'Models: {args.models}')
print(f'Datasets: {len(args.datasets)}')
print(f'Samples per cell: {args.n_samples}')
print(f'Total generations: {total}')
print()

from google.colab import files

for model_key in args.models:
    model_start = time.time()
    print(f'\n========== {model_key} ==========')
    
    # Run one model at a time
    single_args = Args()
    single_args.models = [model_key]
    run_fidelity_experiment(single_args)
    
    elapsed = time.time() - model_start
    print(f'{model_key} completed in {elapsed//60:.0f}m {elapsed%60:.0f}s')
    
    # Save per-model results
    model_file = f'fidelity_results/{model_key}_results.json'
    if os.path.exists(model_file):
        files.download(model_file)
        print(f'Downloaded {model_file}')
    
    # Clean up GPU memory
    import torch
    torch.cuda.empty_cache()

print('\nAll models complete!')

In [ ]:
# Evaluate all results
!python eval_fidelity.py --input fidelity_results/all_results.json --output fidelity_results/summary.json

# Display
import json, IPython.display as disp
with open('fidelity_results/summary.json') as f:
    summary = json.load(f)

html = '<h2>Semantic Fidelity Results</h2>'
html += '<table><tr><th>Model</th><th>Dataset</th><th>n</th><th>ROUGE-L F1</th><th>BERTScore F1</th><th>EM Rate</th></tr>'
for key, entry in sorted(summary['aggregate'].items()):
    model, ds = key.split('/') if '/' in key else (key, '')
    html += f'<tr><td>{model}</td><td>{ds}</td><td>{entry["n"]}</td>'
    html += f'<td>{entry["avg_rougeL_f1"]:.4f}</td>'
    bs = entry.get('avg_bertscore_f1', 'N/A')
    if isinstance(bs, float):
        html += f'<td>{bs:.4f}</td>'
    else:
        html += '<td>N/A</td>'
    html += f'<td>{entry["exact_match_rate"]:.4f}</td></tr>'
html += '</table>'
html += f'<p><b>Overall:</b> {summary["overall"]["n"]} samples, ROUGE-L={summary["overall"]["avg_rougeL_f1"]:.4f}, EM={summary["overall"]["exact_match_rate"]:.4f}</p>'
disp.display(disp.HTML(html))

In [ ]:
from google.colab import files
import zipfile, os

# Zip all results
!zip -r /content/fidelity_results.zip fidelity_results
files.download('/content/fidelity_results.zip')